# Unified model (for dispersing facilities from clients)
This model is a formulation for the **Obnoxious $p$-Median Problem**, specifically the "unified model" for dispersing facilities from clients, as discussed in the paper. The goal is to locate $p$ facilities such that they are as "far" as possible from clients. Unlike traditional median problems that minimize distances, obnoxious facility location maximizes them (or a proxy for them).

$$
\begin{align}
\max \quad & Kt - \sum_{i \in I} u_i \\
\text{s.t.} \quad & t - u_i \leq q_i, \quad \forall i \in I \\
& q_i = Ls_i - \sum_{j \in J} z(i,j), \quad \forall i \in I \\
& s_i - z(i,j) \leq d(i,j) + M(1-y_i), \quad \forall i \in I, j \in J \\
& \sum_{j \in J} y_j = p, \\
& u_i, z(i,j) \geq 0, y_j \in {0,1}, \forall i \in I, j \in J
\end{align}
$$

Here's a breakdown of each part:

**Sets:**
- **$I = \{1, \ldots, n\}$ (Clients):** a set of all demand points or clients that need to be "served" by the facilities.
- **$J = \{1, \ldots, m\}$ (Potential Facility Sites):** a set of all possible locations where a facility can be built.

**Parameters:**
- **$d(i,j)$:** the "distance" or undesirability measure between client $i$ and potential facility site $j$. In obnoxious location, $d(i,j)$ might represent an undesirable impact (e.g., pollution, noise), and larger values are worse for the client.
- **$K$:** a parameter in the objective function. It's related to how many of the "worst" (largest undesirable impact) client-facility assignments are considered.
- **$L$:** a parameter used in constraint (3). It relates to considering the $L$-th smallest distance (or $L$ closest facilities) for each client. In the context of maximizing dispersion, this helps in ensuring facilities are far away.
- **$p$:** the exact number of facilities to be opened.
- **$M$:** a very large positive number (often called a "Big M"). It's used in logical constraints to "turn off" parts of a constraint if a binary variable is zero. $M$ must be large enough to not restrict $s_i - z(i,j)$ when $y_j$ is $0$.

**Decision Variables:**
- **$y_j$ (Binary):** This is the core decision variable.
    - **$y_j = 1$** if a facility is opened at site $j$.
    - **$y_j = 0$** otherwise.
- **$t$ (Continuous):** an auxiliary variable introduced to help linearize the objective function. In this specific formulation, it represents a threshold related to the $K$-th smallest sum of distances, contributing to the maximization of overall dispersion.
- **$u_i$ (Continuous, Non-negative):** auxiliary variables associated with each client $i$. These variables work with $t$ in the objective and constraints to capture the idea of minimizing "undesirable impact."
- **$q_i$ (Continuous):** an auxiliary variable that connects $t$, $u_i$, $s_i$, and $z(i,j)$.
- **$s_i$ (Continuous):** an auxiliary variable for each client $i$. $s_i$ is designed to take the value of the $L$-th smallest distance from client $i$ to any opened facility.
- **$z(i,j)$ (Continuous, Non-negative):** an auxiliary variable. It becomes non-zero if facility $j$ is one of the $L$ closest facilities to client $i$ when facilities are opened.

**Objective Function:**
$$
\begin{align}
\max \quad & Kt - \sum_{i \in I} u_i
\end{align}
$$

The objective is to maximize $K$ times $t$ minus the sum of $u_i$. This structure is common in models that aim to maximize some "worst-case" scenario or dispersion. By maximizing $t$, we are trying to ensure that the $K$-th smallest values (implied by the constraints involving $t$ and $u_i$) are as large as possible, thereby maximizing the "undesirability" for the clients.

**Constraints:**
- constraint (2): **Objective Linkage Constraint** 
$$
t - u_i \leq q_i, \quad \forall i \in I
$$
These constraints link the auxiliary variable $t$ with $u_i$ and $q_i$. They are part of the linearization technique that allows the model to handle the min/max relationships effectively and define $t$ as the $K$-th smallest value related to the sums of distances.
- constraint (3): **L-th Smallest Distance Definition Constraint**
$$
q_i = Ls_i - \sum_{j \in J} z(i,j), \quad \forall i \in I
$$
This constraint relates $q_i$ to $s_i$ and $z(i,j)$. This is key for $s_i$ to represent the $L$-th smallest distance to an opened facility for client $i$.
- constraint (4): **Facility-Client Assignment Logic (Big M) Constrant**
$$
s_i - z(i,j) \leq d(i,j) + M(1-y_j), \quad \forall i \in I, j \in J
$$
This is a "Big M" constraint.
    - If $y_j = 1$ (facility $j$ is open), then $1-y_j = 0$, and the constraint becomes $s_i - z(i,j) \leq d(i,j)$. This forces $s_i$ and $z(i,j)$ to be related to the actual distance $d(i,j)$.
    - If $y_j = 0$ (facility j is not open), then $1-y_j = 1$, and the constraint becomes $s_i - z(i,j) \leq d(i,j) + M$. Since $M$ is very large, this constraint becomes effectively inactive for non-open facilities, allowing $z(i,j)$ to be $0$ (as it should be if the facility isn't open).
- constraint (5): **Cardinality Constraint** or **Number of Facilities Constraint**
$$
\sum_{j \in J} y_j = p
$$
This ensures that exactly $p$ facilities are selected and opened.

## 1. Toy model
Let's create a toy model and walk through it step-by-step.

For our toy model, let's assume:
- **Clients ($I$):** $3$ clients, indexed ${1, 2, 3}$
- **Potential Facility Sites ($J$):** $2$ potential sites, indexed ${1, 2}$
- **Number of Facilities to Open ($p$):** $1$ facility
- **$K$ (Parameter for Objective):** $3$
- **$L$ (Parameter for Constraint):** $1$
- **Distances $d(i,j)$:**
    - $d(1,1) = 10$, $d(1,2) = 5$
    - $d(2,1) = 3$, $d(2,2) = 8$
    - $d(3,1) = 7$, $d(3,2) = 12$

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# Set of clients I
I = [1, 2, 3]  # 3 clients
n = len(I)

# Set of potential facility sites J
J = [1, 2]   # 2 potential sites
m = len(J)

# Parameters
K = n  # Parameter K for the objective, equal to n when the unified model acts exactly as the OpMP model
L = 1  # Parameter L for constraint (3), equal to 1 when the unified model acts exactly as the OpMP model

p = 1  # Number of facilities to open (Constraint (5))

# Distance d(i,j) between client i and facility j
d = {
    (1, 1): 10, (1, 2): 5,
    (2, 1): 3,  (2, 2): 8,
    (3, 1): 7,  (3, 2): 12
}

# A large enough positive number M
# M should be greater than or equal to the maximum possible distance d(i,j).
# For the toy model, max_d is 12, so M=100 is sufficient.
M = 100

In order to plot the clients and the facilities in a graph, I then define a custom function that randomly assigns the 2D coordinates

In [ ]:
import networkx as nx
import random

def build_bipartite_graph(n, m, d, x_range=(0, 100), y_range=(0, 100)):
    
    G = nx.Graph()
    
    # Add client nodes, type='client'
    for i in range(1, n+1):
        G.add_node(f"C{i}", type='client')
    
    # Add facility nodes, type='facility'
    for j in range(1, m+1):
        G.add_node(f"F{j}", type='facility')
    
    # Add edges between clients and facilities with distance attribute
    for (i, j), dist in d.items():
        G.add_edge(f"C{i}", f"F{j}", cost=dist)
    
    # Generate positions for plotting:
    # Clients on y=1 line, spaced by x
    client_pos = {f"C{i}": (generate_random_number(x_range), generate_random_number(y_range)) for i in range(1, n+1)}
    # Facilities on y=0 line, spaced by x
    facility_pos = {f"F{j}": (generate_random_number(x_range), generate_random_number(y_range)) for j in range(1, m+1)}
    pos = {**client_pos, **facility_pos}
    
    # Set position as node attributes
    nx.set_node_attributes(G, pos, 'pos')
    
    return G

def generate_random_number(range, round_to=2):
    return round(random.uniform(*range), round_to)

the I call it to generate the coordinates

In [ ]:
# Build the graph
G = build_bipartite_graph(n, m, d)

In [ ]:
G.nodes.data()

In [ ]:
G.edges.data()

Now, I define the function that plots the graph of the clients and the facilities with their corresponding distances

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

def draw_initial_opmpum_graph(G):

    client_nodes = [n for n, attr in G.nodes(data=True) if attr.get('type') == 'client']
    facility_nodes = [n for n, attr in G.nodes(data=True) if attr.get('type') == 'facility']

    pos = nx.get_node_attributes(G, 'pos')
    
    plt.figure(figsize=(12, 10))
    
    # Draw clients
    nx.draw_networkx_nodes(G, pos,
                           nodelist=client_nodes,
                           node_color='lightblue',
                           node_shape='o',
                           node_size=500,
                           edgecolors='black',
                           label='Clients')
    
    # Draw facilities
    nx.draw_networkx_nodes(G, pos,
                           nodelist=facility_nodes,
                           node_color='gray',
                           node_shape='s',
                           node_size=700,
                           edgecolors='black',
                           label='Facilities')
    
    # Labels with client/facility prefixes
    labels = {node: node for node in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_color='black', font_size=10)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, alpha=0.6)
    
    # Edge labels with costs
    edge_labels = {(u, v): G.edges[u, v]['cost'] for u, v in G.edges()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)
    
    plt.title("Client-Facility Bipartite Graph with Distances")
    plt.axis('off')
    client_handle = mlines.Line2D([], [], color='lightblue', marker='o', linestyle='None',
                                  markersize=10, markeredgecolor='black', label='Clients')
    facility_handle = mpatches.Patch(facecolor='gray', edgecolor='black', label='Facilities')
    
    plt.legend(handles=[client_handle, facility_handle], loc='upper right', borderaxespad=1)
    plt.show()

and I plot the graph

In [ ]:
draw_initial_opmpum_graph(G)

## 2. Gurobi Model creation
Given the unified model (for dispersing facilities from clients) formulation

$$
\begin{align}
\max \quad & Kt - \sum_{i \in I} u_i \\
\text{s.t.} \quad & t - u_i \leq q_i, \quad \forall i \in I \\
& q_i = Ls_i - \sum_{j \in J} z(i,j), \quad \forall i \in I \\
& s_i - z(i,j) \leq d(i,j) + M(1-y_i), \quad \forall i \in I, j \in J \\
& \sum_{j \in J} y_j = p, \\
& u_i, z(i,j) \geq 0, y_j \in {0,1}, \forall i \in I, j \in J
\end{align}
$$

I create the following model

In [ ]:
opmpum = gp.Model("Obnoxious_p_Median_Unified_Model")

## 3. Add Decision Variables
$$
\begin{align}
& t: \quad \text{auxiliary variable} \\
\end{align}

In [ ]:
t = opmpum.addVar(name="t", vtype=GRB.CONTINUOUS)

$$
\begin{align}
u_i &: \text{auxiliary variable for each client } i \\
u_i &= \begin{cases} 
\text{non-zero} & \text{if facility } i \text{ is associated with one of the } K \text{ smallest partial sums} \\
0 & \text{otherwise}
\end{cases}
\end{align}

In [ ]:
u = opmpum.addVars(I, name="u", vtype=GRB.CONTINUOUS, lb=0.0)

$$
\begin{align}
& q_i: \quad \text{intermediate variable for each client } i \\
\end{align}

In [ ]:
q = opmpum.addVars(I, name="q", vtype=GRB.CONTINUOUS)

$$
\begin{align}
& s_i: \quad \text{plays a role similar to } t \\
\end{align}

In [ ]:
s = opmpum.addVars(I, name="s", vtype=GRB.CONTINUOUS)

$$
\begin{align}
z(i,j) &: \text{auxiliary variable} \\
z(i,j) &= \begin{cases} 
\text{non-zero} & \text{when facility } j \text{ is involved in the L closest facilities to client } i \\
0 & \text{otherwise}
\end{cases}
\end{align}

In [ ]:
z = opmpum.addVars(I, J, name="z", vtype=GRB.CONTINUOUS, lb=0.0)

$$
\begin{align}
y_j &: \text{binary variable} \\
y_j &= \begin{cases} 
1 & \text{if facility } j \text{ is opened} \\
0 & \text{otherwise}
\end{cases}
\end{align}

In [ ]:
y = opmpum.addVars(J, name="y", vtype=GRB.BINARY)

## 4. Set the Objective Function
$$
\begin{align}
\max \quad & Kt - \sum_{i \in I} u_i \\
\end{align}
$$

This objective aims to maximize the "dispersion" or "undesirability." $t$ is a variable that is pushed to be as large as possible, essentially representing the $K$-th smallest sum of distances from clients to opened facilities. Subtracting $u_i$ (which are minimized) further pushes the overall value up.

In [ ]:
opmpum.setObjective(K * t - gp.quicksum(u[i] for i in I), GRB.MAXIMIZE)

## 5. Add Constraints
I now add all the constraints of the model

### Objective Linkage Constraint
$$
t - u_i \leq q_i, \quad \forall i \in I
$$

In [ ]:
opmpum.addConstrs((t - u[i] <= q[i] for i in I), name="objective_linkage_constraint")
opmpum.update()

### $L$-th Smallest Distance Definition Constraint
$$
q_i = Ls_i - \sum_{j \in J} z(i,j), \quad \forall i \in I
$$

In [ ]:
opmpum.addConstrs((q[i] == L * s[i] - gp.quicksum(z[i, j] for j in J) for i in I), name="l_th_smallest_distance_definition_constraint")
opmpum.update()

### Facility-Client Assignment Logic (Big M) Constraint
$$
s_i - z(i,j) \leq d(i,j) + M(1-y_i), \quad \forall i \in I, j \in J
$$

In [ ]:
opmpum.addConstrs((s[i] - z[i, j] <= d[i, j] + M * (1 - y[j]) for i in I for j in J), name="facility_client_assignment_logic_constraint")
opmpum.update()

### Cardinality Constraint
$$
\sum_{j \in J} y_j = p
$$

In [ ]:
opmpum.addConstr(gp.quicksum(y[j] for j in J) == p, name="cardinality_constraint")
opmpum.update()

## 6. Optimize the Model

In [ ]:
opmpum.setParam("TimeLimit", 60) # Set a 60 seconds time limit
opmpum.update()
opmpum.write("opmpum.lp")
opmpum.optimize()

## 7. Print Results (if a solution is found)

I now create a function to print the results of the model

In [ ]:
def print_model_solution(model, I, J, t, u, q, s, z, y):
    if model.status == GRB.OPTIMAL:
        print("\n--- Optimal Solution Found ---")
        print(f"Objective value: {model.objVal:.2f}")
        print(f"Variable t: {t.X:.2f}")

        print("\n--- Opened Facilities ---")
        for j in J:
            if y[j].X > 0.5: # Using > 0.5 for binary variables
                print(f"  Facility {j} is opened (y[{j}] = {int(y[j].X)})")

        print("\n--- Auxiliary Variables u_i ---")
        for i in I:
            print(f"  u[{i}]: {u[i].X:.2f}")
        
        print("\n--- Auxiliary Variables q ---")
        for i in I:
            print(f"  q[{i}]: {q[i].X:.2f}")

        print("\n--- Auxiliary Variables s_i ---")
        for i in I:
            print(f"  s[{i}]: {s[i].X:.2f}")
        
        print("\n--- Auxiliary Variables z_(i,j) ---")
        for i in I:
            for j in J:
                print(f"  z({i},{j}): {z[(i,j)].X:.2f}")

    else:
        print(f"\nModel did not solve to optimality. Status: {model.status}")

and immediately use it to print the optimal solution found

In [ ]:
print_model_solution(opmpum, I, J, t, u, q, s, z, y)

I also define a function that plot the solution graph

In [ ]:
def plot_opmpum_solution(G, J, y):

    # Copy graph to avoid modifying original
    H = G.copy()

    # Get open facilities from the Gurobi model
    opened_facilities = [f"F{j}" for j in J if y[j].X > 0.5]
    
    # Identify all facilities
    all_facilities = [n for n, attr in H.nodes(data=True) if attr.get('type') == 'facility']
    
    # Facilities to remove = all except opened
    facilities_to_remove = set(all_facilities) - set(opened_facilities)
    
    # Remove facilities to remove from graph (this also removes their edges)
    H.remove_nodes_from(facilities_to_remove)
    
    # Now call the original plotting function on filtered graph
    draw_initial_opmpum_graph(H)


and then I plot the solution

In [ ]:
plot_opmpum_solution(G, J, y)

## 8. Test correctness
To test my toy model and ensure correctness, I define a brute-force algorithm 

In [ ]:
from itertools import combinations

def brute_force_opmpum(I, J, d, p):
    best_facility_set = None
    best_total_min_distance = -float("inf")
    best_assignment = {}

    # Generate all combinations of p facilities
    for facility_set in combinations(J, p):
        total_min_distance = 0
        assignment = {}

        for i in I:
            # Find nearest facility in the current facility set
            min_dist = float("inf")
            for j in facility_set:
                if d[(i, j)] < min_dist:
                    min_dist = d[(i, j)]
                    assignment[i] = j
            total_min_distance += min_dist

        if total_min_distance > best_total_min_distance:
            best_total_min_distance = total_min_distance
            best_facility_set = facility_set
            best_assignment = assignment.copy()

    return best_facility_set, best_total_min_distance, best_assignment

that I run with the data

In [ ]:
best_facilities, best_obj, assignment = brute_force_opmpum(I, J, d, p)

print("Best facility set (y_j = 1):", best_facilities)
print("Maximum total minimum distance (objective):", best_obj)
print("Client assignments:")
for i in I:
    print(f"  Client {i} → Facility {assignment[i]}")

Now, I define an helper function to automatically compare the Gurobi solution and the brute-force result

In [ ]:
def validate_opmp_unified_model(best_facilities, best_obj, assignment, gurobi_model, t, u, s, q, z, y, I, J, d, p, L, M):

    if gurobi_model.status != GRB.OPTIMAL:
        print("Gurobi did not find an optimal solution.")
        return

    gurobi_obj = gurobi_model.objVal
    gurobi_open_facilities = [j for j in J if y[j].X > 0.5]
    gurobi_open_set = tuple(sorted(gurobi_open_facilities))

    print(f"\nGurobi Objective: {gurobi_obj:.2f}")
    print(f"Brute-force Objective: {best_obj:.2f}")
    print(f"Facilities from Gurobi: {gurobi_open_set}")
    print(f"Facilities from Brute-Force: {best_facilities}")

    # Objective match check
    if abs(gurobi_obj - best_obj) > 1e-6:
        print("Objective mismatch!")
    else:
        print("Objective matches")

    # Facility selection check
    if set(gurobi_open_facilities) != set(best_facilities):
        print("Facility set differs (but may be optimal tie)")
    else:
        print("Facility set matches")

    # Cardinality constraint
    total_y = sum(int(y[j].X + 0.5) for j in J)
    if total_y != p:
        print(f"Cardinality constraint violated: ∑y = {total_y} ≠ {p}")
    else:
        print("Cardinality constraint ∑y = p is satisfied")

    tolerance = 1e-6
    failed = False

    for i in I:
        # Constraint: t - u_i ≤ q_i
        if t.X - u[i].X > q[i].X + tolerance:
            print(f"Constraint violated: t - u[{i}] > q[{i}]")
            failed = True

        # Constraint: q_i = L·s_i - ∑ z(i,j)
        z_sum = sum(z[i, j].X for j in J)
        lhs = q[i].X
        rhs = L * s[i].X - z_sum
        if abs(lhs - rhs) > tolerance:
            print(f"Constraint violated: q[{i}] ≠ L·s[{i}] - ∑z({i},j)")
            failed = True

        for j in J:
            # Constraint: s_i - z(i,j) ≤ d(i,j) + M(1 - y_j)
            left = s[i].X - z[i, j].X
            right = d[(i, j)] + M * (1 - y[j].X)
            if left > right + tolerance:
                print(f"Constraint violated: s[{i}] - z[{i},{j}] > d + M(1 - y)")
                failed = True

    if not failed:
        print("All constraints passed.")

    print("\nBrute-force Assignments:")
    for i in I:
        j = assignment[i]
        print(f"  Client {i} → Facility {j} (distance: {d[i, j]})")


an I call it to ensure correctness

In [ ]:
validate_opmp_unified_model(
    best_facilities, best_obj, assignment, 
    opmpum, t, u, s, q, z, y,
    I=I, J=J, d=d, p=p, L=L, M=M
)

## 9. Paper dataset example
Let's now test the model with an instance given by the reference paper.
First thing first, I create a function to extract the .gz files

In [ ]:
import gzip

def read_gz_file(file_path, encoding='utf-8'):
    try:
        with gzip.open(file_path, 'rt', encoding=encoding) as f:
            content = f.read()
        return content
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return None
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
        return None

In [ ]:
file_content = read_gz_file('obnoxious-p-median-master/data/orlib-cap41.gz')

In [ ]:
file_content

Now I make another function to parse the content of the file

In [ ]:
import re

def parse_obnoxious_data(file_content_string):

    data = {}

    # Extract n, m, p
    n_match = re.search(r'n=(\d+)', file_content_string)
    m_match = re.search(r'm=(\d+)', file_content_string)
    p_match = re.search(r'p=(\d+)', file_content_string)

    if n_match and m_match and p_match:
        data['n'] = int(n_match.group(1))
        data['m'] = int(m_match.group(1))
        data['p'] = int(p_match.group(1))
    else:
        print("Error: Could not find n, m, or p parameters.")
        return None

    # Extract clients
    clients_match = re.search(r'clients = \{(.*?)\}', file_content_string, re.DOTALL)
    if clients_match:
        clients_str = clients_match.group(1).replace('\n', '').replace(' ', '')
        data['clients'] = clients_str.split(',')
    else:
        print("Error: Could not find clients set.")
        return None

    # Extract facilities
    facilities_match = re.search(r'facilities = \{(.*?)\}', file_content_string, re.DOTALL)
    if facilities_match:
        facilities_str = facilities_match.group(1).replace('\n', '').replace(' ', '')
        data['facilities'] = facilities_str.split(',')
    else:
        print("Error: Could not find facilities set.")
        return None

    # Extract table (distances)
    # This regex captures everything from '{' after 'table =' until the final '}' for the table block,
    # being more lenient with internal whitespace and newlines.
    table_match = re.search(r'table = \{\s*((?:.|\s)*?)\s*\}\s*$', file_content_string, re.DOTALL)
    if table_match:
        table_content = table_match.group(1)
        # Remove all whitespace (including newlines) to simplify splitting, then strip leading/trailing braces
        rows_raw = table_content.strip().split('},')
        distance_table = []
        for row_str in rows_raw:
            clean_row_str = row_str.strip(' \n{}') # Clean up any remaining braces or whitespace
            if clean_row_str:
                # Convert each string number to float
                distance_table.append([float(val) for val in clean_row_str.split(',')])
            else:
                # Handle cases where a split might result in an empty string
                continue # Skip empty rows if any
        data['table'] = distance_table
    else:
        print("Error: Could not find table data with the improved regex.")
        return None

    return data

In [ ]:
parsed_data = parse_obnoxious_data(file_content)

In [ ]:
parsed_data

Now I convert the parsed file structure table in a way that I can use with my previous functions

In [ ]:
def convert_parsed_file(parsed_file_data):
    d = {}
    for i in range(len(parsed_file_data['clients'])): # iterate clients
        for j in range(len(parsed_file_data['facilities'])): # iterate facilities
            d[(i+1,j+1)] = parsed_file_data['table'][i][j]
    new_data_structure = {
        'n': parsed_file_data['n'],
        'm': parsed_file_data['m'],
        'p': parsed_file_data['p'],
        'I': [i+1 for i in range(len(parsed_file_data['clients']))],
        'J': [j+1 for j in range(len(parsed_file_data['facilities']))],
        'd': d,
    }
    return new_data_structure

In [ ]:
new_data_structure = convert_parsed_file(parsed_data)

In [ ]:
new_data_structure

Now I build the graph in order to show it

In [ ]:
G = build_bipartite_graph(new_data_structure['n'], new_data_structure['m'], new_data_structure['d'])

In [ ]:
G.nodes.data()

In [ ]:
G.edges.data()

and then I call the function to plot it

In [ ]:
draw_initial_opmpum_graph(G)

Because Gurobi models are not dynamic in the sense that I can just change the data (I, J, d, etc.) and reuse the same model object, to test different instances of the Gurobi model without rewriting the entire model definition, I encapsulate the model building and optimization logic into a function

In [ ]:
def build_and_optimize_opmpum_model(I, J, d, p, M, K=None, L=None, time_limit=60):

    # If we need the unified model to act exactly like the OpMP model
    if(K == None):
        K = len(I)
    
    # If we need the unified model to act exactly like the OpMP model
    if(L == None):
        L = 1

    # Create a new model instance for each call
    opmpum = gp.Model("Obnoxious_p_Median_Unified_Model")

    # Define variables
    t = opmpum.addVar(name="t", vtype=GRB.CONTINUOUS)
    u = opmpum.addVars(I, name="u", vtype=GRB.CONTINUOUS, lb=0.0)
    q = opmpum.addVars(I, name="q", vtype=GRB.CONTINUOUS)
    s = opmpum.addVars(I, name="s", vtype=GRB.CONTINUOUS)
    z = opmpum.addVars(I, J, name="z", vtype=GRB.CONTINUOUS, lb=0.0)
    y = opmpum.addVars(J, name="y", vtype=GRB.BINARY)

    # Set objective function
    opmpum.setObjective(K * t - gp.quicksum(u[i] for i in I), GRB.MAXIMIZE)

    # Add constraints
    opmpum.addConstrs((t - u[i] <= q[i] for i in I), name="objective_linkage_constraint")
    opmpum.update()
    opmpum.addConstrs((q[i] == L * s[i] - gp.quicksum(z[i, j] for j in J) for i in I), name="l_th_smallest_distance_definition_constraint")
    opmpum.update()
    opmpum.addConstrs((s[i] - z[i, j] <= d[i, j] + M * (1 - y[j]) for i in I for j in J), name="facility_client_assignment_logic_constraint")
    opmpum.update()
    opmpum.addConstr(gp.quicksum(y[j] for j in J) == p, name="cardinality_constraint")
    opmpum.update()

    # Set parameters
    opmpum.setParam("TimeLimit", time_limit)
    opmpum.update()

    # Write to file
    opmpum.write("opmpum.lp")
    
    # Optimize the model
    opmpum.optimize()

    return opmpum, t, u, q, s, z, y

Before calling this function, I need to define M. Becuase M should be greater than or equal to the maximum possible distance d(i,j), I create a function that finds the maximum possibile distance and returns it incremented by a custom value like, for example, 100.

In [ ]:
def compute_M(d, padding=100):
    if not d:
        return None  # Handle empty dictionary case
    return max(d.values()) + padding

and I immediately call it

In [ ]:
new_data_structure['M'] = compute_M(new_data_structure['d'])

In [ ]:
new_data_structure['M']

Now I have all the ingredients to finally test the problem given by the paper with the values of the dataset (always given by the paper). For example, with $p=2$

In [ ]:
# Set of clients I
I = new_data_structure['I']
n = len(I)

# Set of potential facility sites J
J = new_data_structure['J']
m = len(J)

# Number of facilities to open (Constraint (5))
p = 2

# Distance d(i,j) between client i and facility j
d = new_data_structure['d']

# A large enough positive number M
M = new_data_structure['M']

In [ ]:
opmpum, t, u, q, s, z, y = build_and_optimize_opmpum_model(I, J, d, p, M)

I then print the data

In [ ]:
print_model_solution(opmpum, I, J, t, u, q, s, z, y)

Finally, I also plot the solution

In [ ]:
plot_opmpum_solution(G, J, y)